# Telco Customer Churn Prediction — MLOps Demo Walkthrough

This notebook walks through the entire MLOps pipeline for predicting customer churn,
from raw data to deployed model with monitoring.

## What We'll Cover

1. **Data Loading** — Load the Telco Customer Churn dataset
2. **Data Quality** — Identify and fix data issues (e.g., TotalCharges blanks)
3. **Data Validation** — Run Great Expectations checks
4. **Feature Engineering** — Preprocess and encode features
5. **Feature Store** — Register and materialize features with Feast
6. **Model Training** — Train with MLflow experiment tracking
7. **Model Registry** — Promote the best model to Production
8. **Model Serving** — Call the FastAPI prediction endpoint
9. **Monitoring** — Detect data drift with Evidently AI

### Prerequisites

Make sure Docker services are running:
```bash
make docker-up
```

## 1. Setup

Import all required libraries. These are installed via `requirements.txt`.

In [ ]:
# === Standard library ===
import os
import sys
import json
import warnings
from datetime import datetime, timedelta

# === Data manipulation ===
import pandas as pd
import numpy as np

# === Visualization ===
import matplotlib.pyplot as plt
import seaborn as sns

# === ML ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    accuracy_score,
)
from xgboost import XGBClassifier

# === MLOps ===
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# === Suppress warnings for cleaner output ===
warnings.filterwarnings("ignore")

# === Add project root to path so we can import src modules ===
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"MLflow version: {mlflow.__version__}")

## 2. Data Loading

Load the Telco Customer Churn dataset from `data/raw/`. This dataset contains
7,043 customer records with 21 columns including demographics, account info,
services subscribed, and the target variable `Churn`.

In [ ]:
# Load the raw dataset
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget distribution:")
print(df["Churn"].value_counts(normalize=True).to_string())

# Preview the first few rows
df.head()

## 3. Data Quality

The Telco dataset has a known data quality issue: the `TotalCharges` column
contains blank strings (not NaN) for customers with `tenure == 0`. These rows
cause type errors if not handled. We also check for other common issues.

In [ ]:
# --- TotalCharges issue ---
# TotalCharges is read as object (string) because some values are blank spaces
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")

# Find the problematic rows — blank strings that cannot be converted to float
blank_total_charges = df[df["TotalCharges"] == " "]
print(f"\nRows with blank TotalCharges: {len(blank_total_charges)}")
print(f"These customers all have tenure == 0:")
print(blank_total_charges[["customerID", "tenure", "MonthlyCharges", "TotalCharges"]].to_string())

# --- Fix: convert to numeric, coerce errors to NaN, then fill with 0 ---
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

print(f"\nAfter fix — TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"Null values remaining: {df['TotalCharges'].isnull().sum()}")

# --- General data quality checks ---
print("\n--- Overall null counts ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n--- Duplicate rows ---")
print(f"Duplicates: {df.duplicated().sum()}")

print("\n--- Numeric summary ---")
df.describe()

## 4. Data Validation with Great Expectations

Great Expectations lets us define and run data quality checks ("expectations")
against the dataset. This is integrated into the Airflow training pipeline
to catch data issues before they reach the model.

In [ ]:
import great_expectations as gx

# Create an ephemeral data context (no persistent GX project needed for demo)
context = gx.get_context()

# Add the dataframe as a data source
data_source = context.data_sources.add_pandas("telco_churn")
data_asset = data_source.add_dataframe_asset(name="telco_df")
batch_definition = data_asset.add_batch_definition_whole_dataframe("full_batch")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

# Define expectations — these mirror what runs in the Airflow DAG
suite = context.suites.add(
    gx.ExpectationSuite(name="telco_churn_suite")
)

# Row count should be reasonable (not empty, not suspiciously large)
suite.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(min_value=1000, max_value=100000)
)

# Required columns must exist
for col in ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]:
    suite.add_expectation(
        gx.expectations.ExpectColumnToExist(column=col)
    )

# Tenure should be non-negative
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="tenure", min_value=0, max_value=200
    )
)

# MonthlyCharges should be positive
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="MonthlyCharges", min_value=0, max_value=500
    )
)

# Churn should only be Yes/No
suite.add_expectation(
    gx.expectations.ExpectColumnDistinctValuesToBeInSet(
        column="Churn", value_set=["Yes", "No"]
    )
)

# Run validation
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="telco_validation",
        data=batch_definition,
        suite=suite,
    )
)

result = validation_definition.run(batch_parameters={"dataframe": df})

# Display results
print(f"Validation success: {result.success}")
for r in result.results:
    status = "PASS" if r.success else "FAIL"
    print(f"  [{status}] {r.expectation_config.type}: {r.expectation_config.kwargs}")

## 5. Feature Engineering

Preprocess the data for model training:
- Encode the binary target `Churn` to 0/1
- Label-encode categorical features
- Scale numeric features
- Create train/test split

This logic lives in `src/data/preprocessing.py` and is called by the Airflow DAG.

In [ ]:
# --- Encode target ---
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# --- Drop customerID (not a feature) ---
df_model = df.drop(columns=["customerID"])

# --- Identify column types ---
categorical_cols = df_model.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = df_model.select_dtypes(include=["number"]).columns.tolist()
numeric_cols.remove("Churn")  # Don't scale the target

print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")

# --- Label encode categoricals ---
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le
    print(f"  {col}: {list(le.classes_)} -> {list(range(len(le.classes_)))}")

# --- Scale numeric features ---
scaler = StandardScaler()
df_model[numeric_cols] = scaler.fit_transform(df_model[numeric_cols])

# --- Train/test split ---
X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]} rows")
print(f"Test set:  {X_test.shape[0]} rows")
print(f"Train churn rate: {y_train.mean():.3f}")
print(f"Test churn rate:  {y_test.mean():.3f}")

## 6. Feature Store with Feast

Feast serves as our feature store, with:
- **Offline store** (file-based): For training — batch retrieval of historical features
- **Online store** (Redis): For serving — low-latency feature lookup at prediction time

The feature definitions live in `config/feast/` and are applied via `make feast-apply`.

In [ ]:
from feast import FeatureStore

# Point to the Feast repo config
FEAST_REPO_PATH = os.path.join(PROJECT_ROOT, "config", "feast")

# Initialize the feature store
# NOTE: This requires `make feast-apply` to have been run first
try:
    store = FeatureStore(repo_path=FEAST_REPO_PATH)

    # List registered feature views
    print("Registered Feature Views:")
    for fv in store.list_feature_views():
        print(f"  - {fv.name}")
        for feature in fv.features:
            print(f"      {feature.name} ({feature.dtype})")

    # Example: retrieve features for a set of customer IDs
    # This demonstrates the offline store (batch retrieval for training)
    # Use IDs from the training parquet (which is the Feast FileSource)
    train_sample = pd.read_parquet(
        os.path.join(PROJECT_ROOT, "data", "processed", "train.parquet"),
        columns=["customerID"],
    ).head(3)

    entity_df = pd.DataFrame({
        "customerID": train_sample["customerID"].tolist(),
        "event_timestamp": pd.to_datetime([datetime.now()] * 3, utc=True),
    })

    # Use the actual feature view names defined in config/feast/features.py
    training_df = store.get_historical_features(
        entity_df=entity_df,
        features=[
            "customer_account:tenure",
            "customer_account:MonthlyCharges",
            "customer_account:TotalCharges",
            "customer_account:Contract",
        ],
    ).to_df()

    print("\nHistorical feature retrieval (offline store):")
    print(training_df.to_string())

    # Example: online feature retrieval (for serving)
    online_features = store.get_online_features(
        features=[
            "customer_account:tenure",
            "customer_account:MonthlyCharges",
        ],
        entity_rows=[{"customerID": train_sample["customerID"].iloc[0]}],
    ).to_dict()

    print("\nOnline feature retrieval (Redis):")
    print(json.dumps(online_features, indent=2, default=str))

except Exception as e:
    print(f"Feast not configured yet. Run 'make feast-apply' first.")
    print(f"Error: {e}")
    print("\nSkipping Feast demo — continuing with preprocessed features.")

## 7. Model Training with MLflow

Train an XGBoost classifier with MLflow experiment tracking. MLflow logs:
- Hyperparameters
- Metrics (accuracy, F1, AUC)
- The trained model artifact
- Confusion matrix plot

MLflow UI is available at http://localhost:5001 when Docker services are running.

In [ ]:
# Connect to the MLflow tracking server running in Docker
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Create or get the experiment
experiment_name = "telco-churn-demo"
mlflow.set_experiment(experiment_name)

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {experiment_name}")

# Define hyperparameters
params = {
    "n_estimators": 200,
    "max_depth": 5,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),  # Handle class imbalance
    "random_state": 42,
    "eval_metric": "logloss",
}

# Train with MLflow tracking
with mlflow.start_run(run_name="xgboost-demo-notebook") as run:
    # Log hyperparameters
    mlflow.log_params(params)

    # Train the model
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    # Generate predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }

    # Log metrics
    mlflow.log_metrics(metrics)

    # Log the model
    mlflow.xgboost.log_model(model, artifact_path="model")

    # Log confusion matrix as artifact
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Churn", "Churn"],
                yticklabels=["No Churn", "Churn"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")
    plt.tight_layout()
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.show()

    # Print results
    print(f"\nRun ID: {run.info.run_id}")
    print(f"\nMetrics:")
    for name, value in metrics.items():
        print(f"  {name}: {value:.4f}")

    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

## 8. Model Registry

MLflow Model Registry tracks model versions and lifecycle stages.
In production, the Airflow DAG automatically registers models that
exceed the F1 threshold, and the FastAPI server loads the latest
Production model on startup.

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

MODEL_NAME = "telco-churn-xgboost"

try:
    # Register the model from the run we just completed
    model_uri = f"runs:/{run.info.run_id}/model"
    model_version = mlflow.register_model(model_uri, MODEL_NAME)

    print(f"Registered model: {MODEL_NAME}")
    print(f"Version: {model_version.version}")
    print(f"Source: {model_version.source}")

    # Set alias (MLflow 2.9+ uses aliases instead of stage transitions)
    client.set_registered_model_alias(MODEL_NAME, "champion", model_version.version)
    print(f"\nSet alias 'champion' -> version {model_version.version}")

    # List all versions
    print(f"\nAll versions of '{MODEL_NAME}':")
    for mv in client.search_model_versions(f"name='{MODEL_NAME}'"):
        print(f"  Version {mv.version} | Status: {mv.status} | Created: {mv.creation_timestamp}")

    # Show aliases
    model_info = client.get_registered_model(MODEL_NAME)
    print(f"\nAliases:")
    for alias in model_info.aliases:
        print(f"  {alias} -> version {client.get_model_version_by_alias(MODEL_NAME, alias).version}")

except Exception as e:
    print(f"Model registry error (MLflow server may not be running): {e}")
    print("Start services with: make docker-up")

## 9. Model Serving

The FastAPI server at http://localhost:8000 loads the Production model
from the MLflow registry and exposes `/predict` and `/health` endpoints.
It also exports Prometheus metrics for monitoring.

In [ ]:
import requests

API_BASE_URL = "http://localhost:8000"

# --- Health check ---
print("=== Health Check ===")
try:
    resp = requests.get(f"{API_BASE_URL}/health", timeout=5)
    print(f"Status: {resp.status_code}")
    print(f"Response: {json.dumps(resp.json(), indent=2)}")
except requests.ConnectionError:
    print("FastAPI server not running. Start with: make docker-up")

# --- Single prediction ---
print("\n=== Single Prediction ===")
payload = {
    "customerID": "7590-VHVEG",
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 1,
    "PhoneService": "No",
    "MultipleLines": "No phone service",
    "InternetService": "DSL",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 29.85,
    "TotalCharges": 29.85,
}

try:
    resp = requests.post(f"{API_BASE_URL}/predict", json=payload, timeout=5)
    print(f"Status: {resp.status_code}")
    result = resp.json()
    print(f"Prediction: {'CHURN' if result.get('churn_prediction') == 1 else 'NO CHURN'}")
    print(f"Probability: {result.get('churn_probability', 'N/A')}")
    print(f"Full response: {json.dumps(result, indent=2)}")
except requests.ConnectionError:
    print("FastAPI server not running. Start with: make docker-up")

# --- Batch prediction (multiple customers) ---
print("\n=== Batch Prediction ===")
# In production, batch predictions are handled by the Airflow DAG
# which writes results to the data warehouse. Here we simulate
# calling the API in a loop.
sample_customers = [
    {**payload, "customerID": "TEST-001", "tenure": 1, "Contract": "Month-to-month"},
    {**payload, "customerID": "TEST-002", "tenure": 48, "Contract": "Two year"},
    {**payload, "customerID": "TEST-003", "tenure": 72, "Contract": "Two year", "MonthlyCharges": 100.0},
]

try:
    for customer in sample_customers:
        resp = requests.post(f"{API_BASE_URL}/predict", json=customer, timeout=5)
        r = resp.json()
        label = "CHURN" if r.get("churn_prediction") == 1 else "NO CHURN"
        prob = r.get("churn_probability", "N/A")
        print(f"  {customer['customerID']}: {label} (p={prob})")
except requests.ConnectionError:
    print("FastAPI server not running. Start with: make docker-up")

## 10. Monitoring with Evidently AI

Evidently detects **data drift** by comparing the distribution of incoming
prediction requests against the training data. When drift is detected in
more than 30% of features, the continuous training DAG is triggered to
retrain the model.

Drift metrics are exported to Prometheus and visualized in the Grafana dashboard.

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

# --- Simulate reference (training) vs current (production) data ---
# In production, current data comes from logged prediction requests
reference_data = X_train.copy()
reference_data["Churn"] = y_train.values

# Simulate drift: shift some feature distributions
current_data = X_test.copy()
current_data["Churn"] = y_test.values

# Add some artificial drift to tenure and MonthlyCharges
# to demonstrate what drift detection looks like
drifted_data = current_data.copy()
drifted_data["tenure"] = drifted_data["tenure"] + np.random.normal(0.5, 0.3, len(drifted_data))
drifted_data["MonthlyCharges"] = drifted_data["MonthlyCharges"] * 1.2

# --- Data Drift Report (clean data) ---
print("=== Data Drift Report (clean data) ===")
drift_report = Report(metrics=[DataDriftPreset()])
snapshot = drift_report.run(
    reference_data=reference_data.drop(columns=["Churn"]),
    current_data=current_data.drop(columns=["Churn"]),
)

# Extract drift results (Evidently 0.7+ dict structure)
drift_results = snapshot.dict()
for m in drift_results["metrics"]:
    if "DriftedColumnsCount" in m["metric_name"]:
        print(f"Drifted columns: {int(m['value']['count'])} ({m['value']['share']:.0%} of features)")
        break

# --- Data Drift Report (with artificial drift) ---
print("\n=== Data Drift Report (with artificial drift) ===")
drift_report_drifted = Report(metrics=[DataDriftPreset()])
snapshot_drifted = drift_report_drifted.run(
    reference_data=reference_data.drop(columns=["Churn"]),
    current_data=drifted_data.drop(columns=["Churn"]),
)

drift_results_drifted = snapshot_drifted.dict()
for m in drift_results_drifted["metrics"]:
    if "DriftedColumnsCount" in m["metric_name"]:
        print(f"Drifted columns: {int(m['value']['count'])} ({m['value']['share']:.0%} of features)")
        break

# Per-feature drift details
print("\nPer-feature drift (drifted data):")
for m in drift_results_drifted["metrics"]:
    if "ValueDrift" in m["metric_name"]:
        col = m["config"].get("column", "unknown")
        threshold = m["config"].get("threshold", 0.05)
        drift_score = m["value"]
        drifted = drift_score < threshold if isinstance(drift_score, (int, float)) else False
        status = "DRIFT" if drifted else "OK"
        if isinstance(drift_score, (int, float)):
            print(f"  [{status}] {col}: p-value={drift_score:.4f}")
        else:
            print(f"  [{status}] {col}: score={drift_score}")

# --- Save HTML report for review ---
report_path = os.path.join(PROJECT_ROOT, "notebooks", "drift_report.html")
snapshot_drifted.save_html(report_path)
print(f"\nDrift report saved to: {report_path}")
print("Open in browser to see interactive visualizations.")